# Glutamate response analysis run

Use this notebook to load one session, run activation/tuning/sequence analyses, and save standardized output tables.

In [1]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    resolve_glutamate_analysis_paths,
    run_glutamate_analysis,
)

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
target_mice = [
    803496,
    804730,
    804733,810196,
    809047,803121,
    826033,838410,834788
]

In [4]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [5]:
assets[0]

SessionAssets(session_id='803496_2025-07-25_13-02-10', subject_id=803496, session_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496'), summary_mat=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08/source_extraction/ExperimentSummary/SummaryLoCo-260117-185357.mat'), bonsai_event_log_csv=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/bonsai_event_log.csv'), harp_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp'), photodiode_pkl=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/803496_2025-07-25_13-02-10/behavior/VCO1_Behavior.harp/extracted_files

In [7]:
for asset in assets[:]:
    session_root = asset.session_dir
    paths = resolve_glutamate_analysis_paths(session_root)
    print(paths)
    
    config = GlutamateAnalysisConfig(
        tuning_method="fve",          # or "manova"
        manova_stat="Wilks' lambda",
        manova_max_timepoints=10,        # reduce from 20
        manova_use_post_only=True,       # much faster, focuses on evoked period
        random_seed=0,
    )

    results = run_glutamate_analysis(session_root, config=config)
    
    {k: v.shape for k, v in results.items() if hasattr(v, 'shape')}
    
    activation_summary = results["activation_summary_table"]
    tuning_per_image = results["tuning_per_image_table"]
    tuning_summary = results["tuning_summary_table"]
    sequence_position = results["sequence_position_table"]
    sequence_per_image = results["sequence_per_image_table"]
    sequence_summary = results["sequence_summary_table"]

    display(activation_summary.head())
    display(tuning_summary.head())
    display(sequence_per_image.head())
    display(sequence_position.head())
    display(sequence_summary.head())

    output_dir = resolve_glutamate_analysis_paths(session_root).output_dir
    print(output_dir)
    print(sorted(p.name for p in output_dir.iterdir()))

GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_sequence_df.npz'), output_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate_analysis'))


,session_id,subject_id,dmd,synapse_id,stimulus_family,n_events,median_delta_auc,mean_delta_auc,effect_direction,p_value,response_class,q_value_within_synapse
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0000,image,2277,-20.015213,-23.085767,none,3.356907e-02,no_change,1.007072e-01
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0001,image,2277,11.062568,10.113008,none,1.885622e-01,no_change,3.431282e-01
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,image,2277,62.247024,75.345806,up,1.833899e-10,activated,5.501698e-10
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,image,2277,51.070708,94.594795,up,5.779422e-15,activated,1.733827e-14
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0004,image,2277,-94.736861,-159.029189,down,1.601894e-34,deactivated,4.805682e-34


,session_id,subject_id,dmd,synapse_id,n_image_trials,n_images_tested,fve_image,p_shuffle_fve,p_kw,p_manova,...,preferred_image,preferred_mean,preferred_median,preferred_vs_rest_effect,preferred_vs_next_effect,is_tuned,tuning_method,q_shuffle_fve,q_kw,q_manova
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,2277,7,0.016428,0.000500,6.161776e-06,1.607803e-06,...,imk01643,209.877494,194.113103,148.201225,79.053404,False,fve,0.000778,1.247760e-05,4.069752e-06
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,2277,7,0.029130,0.000500,2.518969e-11,1.489699e-14,...,imk01378,186.991674,110.479186,69.505918,6.580640,False,fve,0.000778,7.847557e-11,7.541602e-14
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0005,2277,7,0.090994,0.000500,1.177659e-48,3.806244e-44,...,imk01643,622.228322,622.815466,348.406914,133.607561,True,fve,0.000778,1.907808e-47,3.853822e-43
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0006,2277,7,0.007872,0.006997,5.606826e-03,3.841969e-01,...,imk01220,128.009940,143.542635,117.889387,40.459149,False,fve,0.009290,7.569215e-03,3.939234e-01
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0007,2277,7,0.017145,0.000500,1.310140e-05,1.864352e-03,...,imk00895,117.075662,97.514651,58.983192,8.945764,False,fve,0.000778,2.526698e-05,2.849293e-03


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,n_positions,n_sequences,r0,rlast,...,late_mean,early_minus_late,adaptation_index,sequence_slope,sequence_slope_norm,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,21,40,92.015880,-471.204291,...,-384.679330,434.461225,1.000000,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01220.tiff,imk01220,17,40,91.470404,-350.338007,...,99.668370,113.536997,1.000000,-12.382298,-0.135369,64.178829,130.824089,2,response_amplitude,False
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk00459.tiff,imk00459,16,40,29.157173,281.545859,...,167.444344,-120.855572,-0.812315,0.945508,0.032428,-14.799073,63.128745,4,response_amplitude,False
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01643.tiff,imk01643,13,39,87.246938,-96.713450,...,-250.923974,411.061978,1.000000,2.464631,0.028249,156.407801,209.877494,1,response_amplitude,True
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk00942.tiff,imk00942,14,39,117.022731,-180.672781,...,-251.180947,360.275090,1.000000,-36.146476,-0.308884,-96.587349,-6.975492,7,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,position_category,sequence_position,counts,n_sequences,...,response_amplitude_norm,delta_from_r0,r0,sequence_slope,sequence_slope_norm,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,0,40.0,40,...,1.000000,0.000000,92.01588,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,1,40.0,40,...,0.082028,-84.467971,92.01588,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,2,40.0,40,...,2.058219,97.372963,92.01588,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,3,40.0,40,...,0.623534,-34.640865,92.01588,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,stimuli\images_A\imk01378.tiff,imk01378,repeated,4,40.0,40,...,-0.771346,-162.991931,92.01588,-14.972892,-0.162721,-34.922383,45.880193,5,response_amplitude,False


...

,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,n_positions,n_sequences,r0,rlast,...,late_mean,early_minus_late,adaptation_index,sequence_slope,sequence_slope_norm,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,22,40,74.945351,307.801081,...,78.590158,-5.381142,-0.608381,-5.336103,-0.071200,-11.033658,32.570614,5,response_amplitude,False
1,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01097.tiff,imk01097,16,39,71.674961,0.653214,...,141.789856,-65.821748,0.981937,-8.762657,-0.122255,-2.231010,40.115741,4,response_amplitude,False
2,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01057.tiff,imk01057,20,39,223.615671,-181.148947,...,-35.227690,189.314405,1.000000,-9.265998,-0.041437,75.915134,107.098150,1,response_amplitude,True
3,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk00895.tiff,imk00895,17,39,33.051596,282.901875,...,69.530036,-63.635616,-0.790782,-3.443411,-0.104183,-63.416167,-12.328679,7,response_amplitude,False
4,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk00459.tiff,imk00459,20,39,271.016010,-15.859237,...,-3.881859,210.738248,1.000000,-22.871170,-0.084390,35.425682,72.392905,2,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,stimulus_name,stimulus_label,position_category,sequence_position,counts,n_sequences,...,response_amplitude_norm,delta_from_r0,r0,sequence_slope,sequence_slope_norm,image_selectivity_score,ranking_score,image_rank_within_synapse,rank_basis,is_preferred_ranked_image
0,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,repeated,0,40.0,40,...,1.000000,0.000000,74.945351,-5.336103,-0.0712,-11.033658,32.570614,5,response_amplitude,False
1,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,repeated,1,40.0,40,...,0.953664,-3.472671,74.945351,-5.336103,-0.0712,-11.033658,32.570614,5,response_amplitude,False
2,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,repeated,2,40.0,40,...,0.293419,-52.954975,74.945351,-5.336103,-0.0712,-11.033658,32.570614,5,response_amplitude,False
3,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,repeated,3,40.0,40,...,0.595883,-30.286685,74.945351,-5.336103,-0.0712,-11.033658,32.570614,5,response_amplitude,False
4,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,stimuli\images_A\imk01643.tiff,imk01643,repeated,4,40.0,40,...,-0.087629,-81.512710,74.945351,-5.336103,-0.0712,-11.033658,32.570614,5,response_amplitude,False


,session_id,subject_id,dmd,synapse_id,n_images_with_sequences,median_seq_slope,median_adaptation_index,median_r0,median_rlast,median_rterminal,median_terminal_minus_last,median_early_minus_late,seq_p,sequence_class,seq_q
0,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0000,7,-5.336103,0.981937,72.001657,0.653214,123.426447,77.502113,-5.381142,0.015625,stable,0.165179
1,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0004,7,-0.975812,-0.443855,32.065475,83.247913,32.972522,-15.784800,-56.832736,0.218750,stable,0.476103
2,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0010,7,-5.869332,0.023698,207.674063,162.834655,244.382419,22.660666,179.737484,0.046875,stable,0.321181
3,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0011,7,-9.060994,-0.161923,5.151347,-8.897875,52.522036,79.245158,78.021646,0.078125,stable,0.321181
4,803496_2025-07-29_13-34-35,803496,DMD1,DMD1_syn0013,7,1.461472,-0.054164,44.268477,91.342480,49.687171,-32.556083,-14.901660,0.812500,stable,0.897388


\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-29_803496\analysis\derived\glutamate_analysis
['activation_event_table.csv', 'activation_event_table.parquet', 'activation_summary_table.csv', 'activation_summary_table.parquet', 'glutamate_analysis_metadata.json', 'sequence_per_image_table.csv', 'sequence_per_image_table.parquet', 'sequence_position_table.csv', 'sequence_position_table.parquet', 'sequence_summary_table.csv', 'sequence_summary_table.parquet', 'tuning_per_image_table.csv', 'tuning_per_image_table.parquet', 'tuning_summary_table.csv', 'tuning_summary_table.parquet']
GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-30_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-30_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_n

KeyboardInterrupt: 